### Import Library


In [4]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

Load Dataset


In [6]:
df = pd.read_excel(
    "../data/raw/Online Retail.xlsx"
)
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


# Dataset Overview

This section provides basic information about the dataset including the number of rows, columns, and dataset structure.

In [7]:
df.shape

(541909, 8)

## Dataset Size

The dataset contains 541,909 transaction records and 8 attributes.

Each row represents a product transaction recorded by the retail company.

In [8]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


## Initial Findings

The dataset contains 541,909 transaction records with 8 attributes.

The dataset includes transaction information, product details, customer information, and geographic information.

Most columns have appropriate data types. However, CustomerID is detected as float because missing values exist in this column.

# Missing Value Analysis

In [10]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [11]:
(df.isnull().sum() / len(df)) * 100

InvoiceNo       0.000000
StockCode       0.000000
Description     0.268311
Quantity        0.000000
InvoiceDate     0.000000
UnitPrice       0.000000
CustomerID     24.926694
Country         0.000000
dtype: float64

In [12]:
df[df['CustomerID'].notnull()]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


## Missing Value Findings

The dataset contains missing values in two columns: Description and CustomerID.

Description contains 1,454 missing values (0.27%). These records will be removed because product information is required for product performance analysis.

CustomerID contains 135,080 missing values (24.93%). These records will be retained for sales analysis because transactions are still valid. However, records without CustomerID will be excluded during customer behavior analysis.

# Duplicate Analysis

In [13]:
df.duplicated().sum()

np.int64(5268)

In [14]:
(df.duplicated().sum()/len(df))*100

np.float64(0.9721189350979592)

## Duplicate Findings

The dataset contains 5,268 duplicate records, representing approximately 0.97% of the total dataset.

Duplicate records can cause inaccurate sales calculations because the same transaction may be counted multiple times.

Therefore, duplicate rows will be removed during the data cleaning process.

# Transaction Validation Analysis

In [15]:
df['Quantity'].describe()

count    541909.000000
mean          9.552250
std         218.081158
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64

In [16]:
df['UnitPrice'].describe()

count    541909.000000
mean          4.611114
std          96.759853
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64

In [17]:
df[df['InvoiceNo'].astype(str).str.startswith('C')].shape

(9288, 8)

## Transaction Validation Findings

The dataset contains invalid transactions that may affect sales analysis.

### Quantity

The Quantity column contains negative values with a minimum value of -80,995. Negative quantities indicate product returns or cancelled transactions. These records will be removed from sales analysis.


### UnitPrice

The UnitPrice column contains negative values with a minimum value of -11,062.06. Since revenue calculation requires valid selling prices, transactions with UnitPrice <= 0 will be removed.


### Cancelled Transactions

There are 9,288 cancelled transactions identified by InvoiceNo starting with "C". These transactions will be excluded to ensure accurate sales performance analysis.

In [18]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


In [19]:
df['Country'].value_counts().head(10)

Country
United Kingdom    495478
Germany             9495
France              8557
EIRE                8196
Spain               2533
Netherlands         2371
Belgium             2069
Switzerland         2002
Portugal            1519
Australia           1259
Name: count, dtype: int64

# Initial Business Findings


## Sales Data

The dataset contains 541,909 retail transactions.

The average purchased quantity is 9.55 items per transaction. However, negative quantity values were found, indicating cancelled or returned transactions.


## Product Pricing

The average unit price is 4.61. Negative price values were identified and will be removed during the cleaning process.


## Geographic Distribution

The United Kingdom contributes the majority of transactions with 495,478 records, indicating that it is the company's primary market.

Other significant markets include Germany, France, and EIRE.